In [ ]:
import json
from datetime import datetime

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

In [ ]:
load_dotenv()

In [ ]:
model = init_chat_model(model="gpt-4o-mini", temperature=0)

### Tools

In [ ]:
@tool
def get_current_time() -> str:
    """زمان فعلی سیستم را به‌صورت رشته برمی‌گرداند."""
    return datetime.now().strftime("%H:%M:%S")  # noqa: DTZ005

In [ ]:
llm_with_tool = model.bind_tools([get_current_time])

messages = [HumanMessage(content="الان چه ساعتی؟")]
response_without_tool = model.invoke(messages)
response_with_tool = llm_with_tool.invoke(messages)

print("پاسخ اولیه مدل:")
print("نوع:", type(response_with_tool).__name__)
print("محتوا:", response_with_tool.content)
print("\nفراخوانی ابزارها:")
print(json.dumps(response_with_tool.tool_calls, indent=2, ensure_ascii=False))

print("پاسخ اولیه مدل:")
print("نوع:", type(response_without_tool).__name__)
print("محتوا:", response_without_tool.content)
print("\nفراخوانی ابزارها:")
print(json.dumps(response_without_tool.tool_calls, indent=2, ensure_ascii=False))

# tools call Flow
Schema ابزار (که مدل می‌بیند):
{
  "type": "function",
  "function": {
    "name": "get_current_time",
    "description": "زمان فعلی سیستم را به‌صورت رشته برمی‌گرداند.",
    "parameters": {
      "description": "زمان فعلی سیستم را به‌صورت رشته برمی‌گرداند.",
      "properties": {},
      "title": "get_current_time",
      "type": "object"
    }
  }
}

In [ ]:
def run_tool_loop(llm_with_tools, messages, tools_map, max_turns=5):
    """
    حلقه دستی tool calling:
    مدل → درخواست ابزار → اجرای ابزار → ارسال نتیجه → پاسخ نهایی

    Args:
        llm_with_tools: مدل با ابزارهای متصل
        messages: لیست پیام‌ها
        tools_map: دیکشنری {نام_ابزار: تابع_ابزار}
        max_turns: حداکثر دور تکرار

    Returns:
        پیام نهایی AIMessage
    """
    print("─" * 50)
    for turn in range(max_turns):
        # فراخوانی مدل
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # اگر مدل ابزاری نخواست، پاسخ نهایی آماده است
        if not response.tool_calls:
            print(f"🏁 پاسخ نهایی (دور {turn+1}):")
            print(response.content)
            break

        # اجرای هر ابزار درخواستی
        print(f"🔄 دور {turn+1}: مدل {len(response.tool_calls)} ابزار درخواست کرد")
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]

            print(f"  🔧 اجرای ابزار: {tool_name}({tool_args})")

            # اجرای تابع محلی
            if tool_name in tools_map:
                result = tools_map[tool_name].invoke(tool_args)
            else:
                result = f"خطا: ابزار '{tool_name}' یافت نشد"

            print(f"  ✅ نتیجه: {result}")

            # ارسال نتیجه ابزار به مدل
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_id)
            )
        print("─" * 50)

    return messages[-1].content

In [ ]:
# نگاشت نام ابزار به تابع
tools_map = {"get_current_time": get_current_time}

# اجرا
messages = [HumanMessage(content="الان چه ساعتیه؟")]
final_response = run_tool_loop(llm_with_tool, messages, tools_map)

### Nice Tools Sample

In [ ]:
import qrcode
import requests
from qrcode.image.styledpil import StyledPilImage


@tool
def get_weather_from_ip() -> str:
    """
    Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it to the user.
    """

    return "Current: 12.5°C, High: 13.0°C, Low: 0.3°C"
    # Get location coordinates from the IP address
    #ip_info = requests.get('https://ip-api.com/json/').json()
    #lat, lon = ip_info['lat'], ip_info['lon']

    ip_info = requests.get('https://geolocation-db.com/json/').json()
    lat, lon = ip_info['latitude'], ip_info['longitude']



    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # Format and return the simplified string
    return (
        f"Current: {weather_data['current']['temperature_2m']}°C, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°C, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°C"
    )


@tool
def write_text_to_file(file_path:str,content:str)->str:
    """یک فایل متنی (.txt) با محتوای مشخص می‌سازد یا بازنویسی می‌کند.

    Args:
        file_path: مسیر فایل مقصد (مثلاً reminder.txt)
        content: محتوای متنی که در فایل نوشته می‌شود
    """

    with open(file_path,"w",encoding="utf-8") as f:
        f.write(content)
    return f"فایل {file_path} با موفقیت ذخیره شد."

@tool
def generate_qr_code(data: str, filename: str, image_path: str) -> str:
    """یک کد QR با لوگوی تعبیه‌شده تولید می‌کند و به‌صورت PNG ذخیره می‌کند.

    Args:
        data: متن یا URL برای رمزگذاری در QR
        filename: نام فایل خروجی (بدون پسوند .png)
        image_path: مسیر تصویر لوگو برای تعبیه در مرکز QR
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)
    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)
    return f"کد QR در فایل '{output_file}' ذخیره شد | محتوا: {data[:50]}"


print("✅ ابزارها تعریف شدند")
print(f"  - {get_weather_from_ip.name}: {get_weather_from_ip.description}")
print(f"  - {write_text_to_file.name}: {write_text_to_file.description}")
print(f"  - {generate_qr_code.name}: {generate_qr_code.description}")

In [ ]:
all_tools = [
    get_current_time,
    get_weather_from_ip,
    write_text_to_file,
    generate_qr_code
]

all_tools_map = {t.name: t for t in all_tools}

llm_with_all_tools = model.bind_tools(all_tools)

print(f"✅ {len(all_tools)} ابزار به مدل متصل شد:")

for t in all_tools:
    print(f"  • {t.name}")

In [ ]:
print("🌤️ آزمایش ۱: پرسش درباره آب‌وهوا")
print("="*50)

messages = [HumanMessage(content="می‌تونی وضعیت آب‌وهوا رو برای محل فعلی من بگی؟")]
response = run_tool_loop(llm_with_tools=llm_with_all_tools, messages=messages, tools_map=all_tools_map)

In [ ]:
# آزمایش ۲: ساختن فایل یادآوری
print("📝 آزمایش ۲: ساختن فایل یادآوری")
print("="*50)

messages = [HumanMessage(
content="30 آبان نوبت دکتر دارم میخوام یادم نره"
)]
response = run_tool_loop(llm_with_all_tools, messages, all_tools_map)

In [ ]:
# آزمایش ۳: تولید QR Code
print("📱 آزمایش ۳: تولید کد QR")
print("="*50)

messages = [HumanMessage(
    content="یه QR code برام بساز که به https://arash.click/ لینک بده. لوگوی من در فایل Arash.png هست."
)]

response = run_tool_loop(llm_with_all_tools, messages, all_tools_map)

In [15]:
print("🔗 آزمایش چند ابزاری")
print("="*50)

prompt = "میخوام آب و هوای فردای اشترینان رو واسم بنویسی یادم نره"
messages = [HumanMessage(content=prompt)]
response = run_tool_loop(llm_with_all_tools, messages, all_tools_map, max_turns=10)

🔗 آزمایش چند ابزاری
──────────────────────────────────────────────────
🔄 دور 1: مدل 1 ابزار درخواست کرد
  🔧 اجرای ابزار: get_weather_from_ip({})
  ✅ نتیجه: Current: 12.5°C, High: 13.0°C, Low: 0.3°C
──────────────────────────────────────────────────
🔄 دور 2: مدل 1 ابزار درخواست کرد
  🔧 اجرای ابزار: write_text_to_file({'content': 'آب و هوای فردای اشترینان:\nدمای فعلی: 12.5°C\nحداکثر دما: 13.0°C\nحداقل دما: 0.3°C', 'file_path': 'weather_ashtarin.txt'})
  ✅ نتیجه: فایل weather_ashtarin.txt با موفقیت ذخیره شد.
──────────────────────────────────────────────────
🏁 پاسخ نهایی (دور 3):
آب و هوای فردای اشترینان با موفقیت در فایل "weather_ashtarin.txt" ذخیره شد. اگر نیاز به اطلاعات بیشتری دارید، خوشحال می‌شوم کمک کنم!
